In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load the dataset
data = pd.read_csv("iris.csv", header='infer').values

# Extract features and labels
X = data[:, :-1]  # All columns except the last one
y, y_labels = np.unique(data[:, -1], return_inverse=True)  # Convert string labels to numbers

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, stratify=y_labels, random_state=42)

# Number of classes
nclasses = np.unique(y_train).shape[0]

# Get k value from user
k = int(input("Enter the number of nearest neighbours to be used, i.e. k: "))

# Prediction array
pred = np.zeros(X_test.shape[0], dtype=int)

# KNN Algorithm
for i in range(X_test.shape[0]):
    dist = np.sqrt(np.sum((X_train - X_test[i]) ** 2, axis=1))  # Euclidean distance
    kminind = np.argsort(dist)[:k]  # Find k nearest neighbors
    
    invdist = 1 / (dist + 1e-10)  # Avoid divide by zero
    denom = np.sum(invdist[kminind])  # Normalization factor
    
    classvotes = np.zeros(nclasses)  # Array for votes
    
    # Voting mechanism
    for j in range(k):
        classvotes[y_train[kminind[j]]] += invdist[kminind[j]]
    
    classvotes /= denom  # Normalize
    pred[i] = np.argmax(classvotes)  # Pick class with max votes

# Define performance metrics
def calc_acc(y_pred, y_true):
    return np.sum(y_pred == y_true) / y_pred.shape[0]

def calc_prec(y_pred, y_true):
    classes = np.unique(y_true)
    nclasses = classes.shape[0]
    precclasswise = np.zeros(nclasses)
    
    for i in range(nclasses):
        pred_indices = np.where(y_pred == classes[i])[0]
        true_indices = np.where(y_true == classes[i])[0]
        
        if len(pred_indices) == 0:
            precclasswise[i] = 0  # Avoid division by zero
        else:
            precclasswise[i] = len(set(pred_indices) & set(true_indices)) / len(pred_indices)
    
    return np.mean(precclasswise)  # Macro-averaged precision

def calc_recall(y_pred, y_true):
    classes = np.unique(y_true)
    nclasses = classes.shape[0]
    recallclasswise = np.zeros(nclasses)
    
    for i in range(nclasses):
        pred_indices = np.where(y_pred == classes[i])[0]
        true_indices = np.where(y_true == classes[i])[0]
        
        if len(true_indices) == 0:
            recallclasswise[i] = 0  # Avoid division by zero
        else:
            recallclasswise[i] = len(set(pred_indices) & set(true_indices)) / len(true_indices)
    
    return np.mean(recallclasswise)  # Macro-averaged recall

# Calculate performance metrics
accuracy = calc_acc(pred, y_test)
precision = calc_prec(pred, y_test)
recall = calc_recall(pred, y_test)

if (precision + recall) > 0:
    f1score = (2 * precision * recall) / (precision + recall)
else:
    f1score = 0  # Avoid divide by zero

# Display results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1score)

# Validate results using sklearn
print("\nClassification Report:")
print(classification_report(y_test, pred, target_names=y))


NameError: name 'mat' is not defined